# Predicciones baseline de Qwen3.5-2B sobre Reddit

Este notebook ejecuta **el modelo base sin cargar el adaptador LoRA** y guarda sus predicciones por separado. Así puedes comparar posteriormente el baseline contra el modelo ajustado.

- No entrena ni modifica el adaptador.
- Usa el mismo `INSTRUCCION_CHAT`, plantilla de conversación y estrategia greedy del notebook de entrenamiento.
- Guarda un CSV, un JSONL con los metadatos originales y un JSON con métricas comparables.
- `BLEU` queda como `null` porque el JSON de Reddit no contiene referencias neutralizadas.
- BERTScore se calcula contra la frase original de entrada; no debe interpretarse como una métrica de calidad de neutralización.

In [1]:
# 1. Paquetes y configuración del entorno
import os
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import importlib.metadata as md

paquetes = ["torch", "transformers", "pandas", "bert-score"]
faltan = []
for p in paquetes:
    try:
        print(f"{p:14s} {md.version(p)}")
    except md.PackageNotFoundError:
        faltan.append(p)
        print(f"{p:14s} NO instalado")

if faltan:
    raise RuntimeError(
        "Faltan paquetes. Ejecuta: %pip install -U "
        + " ".join(faltan)
        + " ; luego reinicia el kernel."
    )


torch          2.14.0+cu130
transformers   5.18.0
pandas         3.0.5
bert-score     0.3.13


In [2]:
# 2. Configuración de rutas y modelo
from pathlib import Path
import json, re, sys, gc, unicodedata
import torch

MODELO_ID = "Qwen/Qwen3.5-2B"
MAX_NEW_TOKENS = 96
BATCH_SIZE = 8
SEED = 42

# Busca la raíz del repositorio.
marca = "data/entrenamiento/splits/split_oficial.csv"
actual = Path.cwd().resolve()
RAIZ = next((d for d in [actual, *actual.parents] if (d / marca).is_file()), None)
assert RAIZ is not None, (
    f"No encontré {marca} subiendo desde {actual}. "
    "Abre este notebook desde una carpeta dentro de NLP-ChiNeutro."
)

# Ajusta esta ruta si tu JSON de Reddit está en otra ubicación.
RUTA_REDDIT = RAIZ / "data" / "reddit" / "reddit.json"

# El baseline queda guardado separado de las predicciones LoRA.
DIR_PRED = RAIZ / "models" / "Qwen3.5" / "2B" / "predicciones" / "reddit"
DIR_PRED.mkdir(parents=True, exist_ok=True)

assert RUTA_REDDIT.is_file(), (
    f"No existe el JSON de Reddit: {RUTA_REDDIT}\n"
    "Corrige RUTA_REDDIT en esta celda."
)
assert torch.cuda.is_available(), (
    "No se detectó CUDA. Este notebook está preparado para inferencia local con GPU."
)

BF16 = torch.cuda.get_device_capability(0)[0] >= 8
DTYPE = torch.bfloat16 if BF16 else torch.float16

print("Repo:", RAIZ)
print("Modelo baseline:", MODELO_ID)
print("Dataset Reddit:", RUTA_REDDIT)
print("Salidas:", DIR_PRED)
print("GPU:", torch.cuda.get_device_name(0), "| dtype:", DTYPE)


Repo: C:\Users\conso\OneDrive\Desktop\Universidah\2026-2\NLP\Proyecto\NLP-ChiNeutro
Modelo baseline: Qwen/Qwen3.5-2B
Dataset Reddit: C:\Users\conso\OneDrive\Desktop\Universidah\2026-2\NLP\Proyecto\NLP-ChiNeutro\data\reddit\reddit.json
Salidas: C:\Users\conso\OneDrive\Desktop\Universidah\2026-2\NLP\Proyecto\NLP-ChiNeutro\models\Qwen3.5\2B\predicciones\reddit
GPU: NVIDIA GeForce RTX 5070 | dtype: torch.bfloat16


In [3]:
# 3. Leer el dataset de Reddit
with open(RUTA_REDDIT, "r", encoding="utf-8") as f:
    data_reddit = json.load(f)

assert "frases" in data_reddit, "El JSON no contiene la clave 'frases'."
frases_reddit = data_reddit["frases"]
assert isinstance(frases_reddit, list) and frases_reddit, "La lista 'frases' está vacía o no es válida."

for i, item in enumerate(frases_reddit):
    assert "frase" in item, f"El registro {i} no contiene 'frase'."

print("Metadatos:", data_reddit.get("meta", {}))
print("Cantidad de frases:", len(frases_reddit))
print(json.dumps(frases_reddit[0], ensure_ascii=False, indent=2)[:2500])


In [4]:
# 4. Cargar el mismo prompt usado durante el entrenamiento
sys.path.insert(0, str(RAIZ))
sys.path.insert(0, str(RAIZ / "models" / "gemma4"))
from comun import INSTRUCCION_CHAT

def entrada(frase):
    return INSTRUCCION_CHAT.format(chilena=frase.strip())

from transformers import AutoTokenizer, set_seed

set_seed(SEED)
tok = AutoTokenizer.from_pretrained(MODELO_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

FIN_TURNO = "<|im_end|>"
ID_FIN = tok.convert_tokens_to_ids(FIN_TURNO)
assert ID_FIN is not None, f"El tokenizador no tiene {FIN_TURNO}"

def prompt_de(frase, pensar=False):
    mensajes = [{"role": "user", "content": entrada(frase)}]
    return tok.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=pensar
    )

tok.padding_side = "left"
print(prompt_de(frases_reddit[0]["frase"]))


In [5]:
# 5. Cargar Qwen3.5-2B SIN adaptador LoRA
from transformers import AutoModelForCausalLM

modelo = AutoModelForCausalLM.from_pretrained(
    MODELO_ID,
    dtype=DTYPE,
    device_map={"": 0}
)
modelo.eval()
modelo.config.use_cache = True

print(type(modelo).__name__)
print(f"Memoria GPU asignada: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Qwen3_5ForCausalLM
Memoria GPU asignada: 3.76 GB


In [6]:
# 6. Función de generación (greedy, misma configuración que el notebook de Qwen)
@torch.no_grad()
def traducir(frases, bs=BATCH_SIZE):
    modelo.eval()
    salidas = []
    for i in range(0, len(frases), bs):
        lote = frases[i:i + bs]
        x = tok(
            [prompt_de(f) for f in lote],
            return_tensors="pt",
            padding=True,
            add_special_tokens=False
        ).to(modelo.device)

        out = modelo.generate(
            **x,
            do_sample=False,
            num_beams=1,
            temperature=None,
            top_p=None,
            top_k=None,
            max_new_tokens=MAX_NEW_TOKENS,
            use_cache=True,
            eos_token_id=list({ID_FIN, tok.eos_token_id} - {None}),
            pad_token_id=tok.pad_token_id
        )

        textos = tok.batch_decode(
            out[:, x["input_ids"].shape[1]:],
            skip_special_tokens=True
        )
        for s in textos:
            s = re.sub(r"<think>.*?</think>", "", s, flags=re.S)
            salidas.append(s.strip().split("\n")[0].strip())
    return salidas

# Prueba de funcionamiento con hasta tres frases.
muestra = [x["frase"] for x in frases_reddit[:min(3, len(frases_reddit))]]
for original, pred in zip(muestra, traducir(muestra, bs=min(3, len(muestra)))):
    print("ENTRADA:   ", original)
    print("BASELINE:  ", pred)
    print("---")


In [7]:
# 7. Generar y guardar todas las predicciones baseline
entradas_reddit = [x["frase"].strip() for x in frases_reddit]
predicciones_baseline = traducir(entradas_reddit)

assert len(predicciones_baseline) == len(frases_reddit), (
    "La cantidad de predicciones no coincide con Reddit."
)

import pandas as pd

filas = []
for item, pred in zip(frases_reddit, predicciones_baseline):
    filas.append({
        "id_mistral": item.get("id", ""),
        "frase_chilena": item["frase"],
        "prediccion": pred
    })

df_baseline = pd.DataFrame(
    filas,
    columns=["id_mistral", "frase_chilena", "prediccion"]
)

ARCHIVO_CSV = DIR_PRED / "predicciones_Qwen3_5_2B_baseline_reddit.csv"
df_baseline.to_csv(ARCHIVO_CSV, index=False, encoding="utf-8")

ARCHIVO_DETALLE = DIR_PRED / "predicciones_Qwen3_5_2B_baseline_reddit_detalle.jsonl"
with open(ARCHIVO_DETALLE, "w", encoding="utf-8") as f:
    for item, pred in zip(frases_reddit, predicciones_baseline):
        registro = dict(item)
        registro["prediccion_baseline"] = pred
        f.write(json.dumps(registro, ensure_ascii=False) + "\n")

print("CSV:", ARCHIVO_CSV)
print("Detalle:", ARCHIVO_DETALLE)
print("Predicciones:", len(df_baseline))
print("Vacías:", int((df_baseline["prediccion"].str.strip() == "").sum()))
display(df_baseline.head(10))


In [8]:
# 8. Métrica de términos retenidos (misma definición del notebook de Reddit LoRA)
def normalizar(texto):
    texto = str(texto or "").casefold().strip()
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    texto = re.sub(r"[^a-z0-9ñ]+", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()

total_expresiones = 0
expresiones_retenidas = 0

for item, pred in zip(frases_reddit, predicciones_baseline):
    pred_norm = normalizar(pred)
    for exp in item.get("expresiones", []) or []:
        termino = exp.get("forma") or exp.get("clave") or exp.get("termino")
        if not termino:
            continue
        total_expresiones += 1
        termino_norm = normalizar(termino)
        if termino_norm and re.search(
            r"(?<!\w)" + re.escape(termino_norm) + r"(?!\w)",
            pred_norm
        ):
            expresiones_retenidas += 1

terminos_retenidos = (
    100.0 * expresiones_retenidas / total_expresiones
    if total_expresiones else None
)

print("Expresiones anotadas:", total_expresiones)
print("Expresiones retenidas:", expresiones_retenidas)
print("Términos retenidos (%):", terminos_retenidos)


Expresiones anotadas: 2424
Expresiones retenidas: 1742
Términos retenidos (%): 71.86468646864687


In [9]:
# 9. Calcular BERTScore contra la frase original
from bert_score import score as bertscore_score

P, R, F1 = bertscore_score(
    predicciones_baseline,
    entradas_reddit,
    lang="es",
    verbose=True,
    batch_size=16
)
bertscore_f1 = float(F1.mean().item())
print("BERTScore F1 contra entrada:", bertscore_f1)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/223 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/119 [00:00<?, ?it/s]

done in 2.93 seconds, 648.88 sentences/sec
BERTScore F1 contra entrada: 0.9299888610839844


In [10]:
# 10. Guardar resumen JSON con esquema compatible con la evaluación de Gemma
n = len(frases_reddit)

resumen_baseline = {
    "n": n,
    "bleu": None,
    "terminos_retenidos_%": round(terminos_retenidos, 4) if terminos_retenidos is not None else None,
    "bertscore_f1": round(bertscore_f1, 4),
    "bertscore_contra": "entrada",
    "juez_modelo": None,
    "juez_effort": None,
    "juez_rubrica": None,
    "juez_n": 0,
    "juez_faltan": n,
    "juez_sentido_media": None,
    "juez_fluidez_media": None,
    "juez_jerga_eliminada_%": None,
    "juez_aceptable_%": None
}

ARCHIVO_EVALUACION = DIR_PRED / "predicciones_Qwen3_5_2B_baseline_reddit_evaluacion.json"
with open(ARCHIVO_EVALUACION, "w", encoding="utf-8") as f:
    json.dump(resumen_baseline, f, ensure_ascii=False, indent=2)

print(json.dumps(resumen_baseline, ensure_ascii=False, indent=2))
print("\nGuardado:", ARCHIVO_EVALUACION)


{
  "n": 1900,
  "bleu": null,
  "terminos_retenidos_%": 71.8647,
  "bertscore_f1": 0.93,
  "bertscore_contra": "entrada",
  "juez_modelo": null,
  "juez_effort": null,
  "juez_rubrica": null,
  "juez_n": 0,
  "juez_faltan": 1900,
  "juez_sentido_media": null,
  "juez_fluidez_media": null,
  "juez_jerga_eliminada_%": null,
  "juez_aceptable_%": null
}

Guardado: C:\Users\conso\OneDrive\Desktop\Universidah\2026-2\NLP\Proyecto\NLP-ChiNeutro\models\Qwen3.5\2B\predicciones\reddit\predicciones_Qwen3_5_2B_baseline_reddit_evaluacion.json


## Archivos generados

En `models/Qwen3.5/2B/predicciones/reddit/`:

- `predicciones_Qwen3_5_2B_baseline_reddit.csv`: predicciones en formato tabular.
- `predicciones_Qwen3_5_2B_baseline_reddit_detalle.jsonl`: predicciones junto con los metadatos originales de cada frase.
- `predicciones_Qwen3_5_2B_baseline_reddit_evaluacion.json`: métricas automáticas y campos del juez pendientes.

**Importante:** este notebook evalúa el baseline (modelo base sin LoRA). Para una comparación justa, ejecuta el notebook LoRA sobre el mismo JSON de Reddit. Los campos del juez permanecen vacíos porque aquí no se ha conectado el evaluador externo.